# 12. Query Understanding Golden Set Design

## 1. 분석 목적

11차 Rule-based Baseline
(미리 정한 명확한 규칙으로 자연어를 Feature로 변환하는 가장 단순한 기준 모델)은 126개 Synthetic Query에서
Parser Exact Match 100%를 기록했다. 하지만 그 문장은 현재 규칙과 Template을 바탕으로 만들었으므로 실제
사용자의 자유로운 자연어 이해 성능을 뜻하지 않는다.

이번 Notebook의 목적은 실제 사람이 자유롭게 쓴 Query를 평가할 **Query Understanding Golden Set**
(모델이 맞았는지 평가하기 위해 사람이 정답을 붙인 데이터)으로 수집하기 위한 구조와 도구를 설계하는 것이다.

이번 단계에서는 추천 향수 정답을 만들지 않는다. 먼저 다음 변환만 독립적으로 평가한다.

> Natural Language → Structured Feature


## 2. Query Understanding과 Recommendation을 분리하는 이유

### Stage 1 — Query Understanding

사용자: “여름 밤에 쓰기 좋은 달콤한 바닐라 향”

↓

Accord / Note / Season / Daypart / Unresolved semantic expression으로 구조화

### Stage 2 — Recommendation

구조화된 Target Profile

↓

향수 Ranking

두 단계를 분리해야 추천 결과가 나쁠 때 원인이 문장 이해 오류인지, 문장을 올바르게 이해한 뒤의 Ranking
오류인지 구분할 수 있다. 이번 Golden Set은 **Stage 1만 평가**한다.

평가 대상:

- Accord, Note, Season, Daypart 추출
- 해석하지 못해야 하는 표현 탐지
- Partial Resolution(한 문장의 일부만 이해하고 일부는 해석하지 못한 상태) 탐지

평가하지 않는 대상:

- 실제 추천 향수 만족도와 Ranking 품질
- TOP 10 정확도와 구매 가능성


## 3. Evaluation Leakage 방지

Evaluation Leakage
(평가 데이터를 만들 때 모델의 규칙을 미리 보고 문장을 작성해 평가가 지나치게 쉬워지는 문제)를 막기 위해
Query 작성자는 다음을 보지 않는다.

- `11_rule_lexicon.csv`
- 11차 Parser 코드와 Rule 목록
- `11_parser_test_results.csv`의 126개 Synthetic Query

이 Notebook도 위 파일들을 로드하지 않는다. Query 작성자는 별도의 블라인드 작업 화면에서 자유롭게 문장을
작성하고, Annotation 담당자가 그 다음 단계에서 직접 대응 가능한 Feature와 unresolved 표현을 표시한다.
기존 Synthetic Query는 개발·동작 확인용이며 새 Golden Set에 복사하지 않는다.


## 4. Query 유형과 120개 빈 작성 슬롯

실제 문장은 자동 생성하지 않는다. Notebook은 권장 범위 120~150개 안에서 **120개의 빈 슬롯**만 만든다.

| 표준 `query_type` | 의미 | 목표 비율 | 120개 기준 |
|---|---|---:|---:|
| `DIRECT` | Accord / Note를 직접 말함 | 20% | 24 |
| `CONTEXT` | Season / Daypart를 명시 | 15% | 18 |
| `SENSORY` | 감각적 형용사 중심 | 20% | 24 |
| `SCENE_ABSTRACT` | 장면·분위기 중심 | 20% | 24 |
| `MIXED` | 직접 표현과 어려운 표현이 섞임 | 25% | 30 |

`query_id`와 유형만 미리 배정하고 `query_text`와 모든 Gold label은 비워 둔다. 빈 문자열은 “아직 작성·
Annotation하지 않음”을 뜻하며 빈 집합 `[]`과 구분된다.


In [1]:
import itertools
import json
import math
import pathlib
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 130)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

work_dir = pathlib.Path.cwd()
output_dir = work_dir / "analysis_outputs"
output_dir.mkdir(exist_ok=True)

annotation_template_path = output_dir / "12_query_annotation_template.csv"
annotations_long_path = output_dir / "12_query_annotations_long.csv"
goldset_path = output_dir / "12_query_goldset.csv"
guideline_path = output_dir / "12_annotation_guideline.md"
prediction_path = output_dir / "12_rule_parser_predictions.csv"
evaluation_path = output_dir / "12_rule_parser_gold_evaluation.csv"
errors_path = output_dir / "12_rule_parser_errors.csv"

ANNOTATION_COLUMNS = [
    "query_id", "query_text", "query_type",
    "gold_accords", "gold_notes", "gold_seasons", "gold_dayparts",
    "gold_unresolved_expressions", "resolution_status",
    "annotator_confidence", "annotator_comment",
]
LONG_COLUMNS = [
    "query_id", "annotator_id",
    "gold_accords", "gold_notes", "gold_seasons", "gold_dayparts",
    "gold_unresolved_expressions", "resolution_status",
    "annotator_confidence", "annotator_comment",
]
GOLDSET_COLUMNS = [
    "query_id", "query_text", "query_type",
    "gold_accords", "gold_notes", "gold_seasons", "gold_dayparts",
    "gold_unresolved_expressions", "resolution_status",
    "agreement_score", "review_status",
]

QUERY_TYPE_QUOTAS = {
    "DIRECT": 24,
    "CONTEXT": 18,
    "SENSORY": 24,
    "SCENE_ABSTRACT": 24,
    "MIXED": 30,
}
ALLOWED_QUERY_TYPES = tuple(QUERY_TYPE_QUOTAS)
ALLOWED_RESOLUTION_STATUS = (
    "FULLY_RESOLVABLE", "PARTIALLY_RESOLVABLE", "UNRESOLVABLE"
)
ALLOWED_CONFIDENCE = ("HIGH", "MEDIUM", "LOW")
ALLOWED_REVIEW_STATUS = ("READY", "MANUAL_REVIEW", "EXCLUDED")


## 5. Annotation Template 생성

Annotation
(사람이 데이터에 정답이나 의미 정보를 직접 표시하는 작업) 파일은 재실행 시 사람의 입력을 덮어쓰지 않는다.
파일이 이미 있으면 column 구조만 검증하고 그대로 읽는다.


In [2]:
def validate_columns(frame, expected_columns, label):
    missing = [column for column in expected_columns if column not in frame.columns]
    if missing:
        raise ValueError(f"{label}에 필요한 column이 없습니다: {missing}")


def create_blank_query_slots():
    rows = []
    index = 1
    for query_type, count in QUERY_TYPE_QUOTAS.items():
        for _ in range(count):
            rows.append({
                "query_id": f"GQ{index:04d}",
                "query_text": "",
                "query_type": query_type,
                "gold_accords": "",
                "gold_notes": "",
                "gold_seasons": "",
                "gold_dayparts": "",
                "gold_unresolved_expressions": "",
                "resolution_status": "",
                "annotator_confidence": "",
                "annotator_comment": "",
            })
            index += 1
    return pd.DataFrame(rows, columns=ANNOTATION_COLUMNS)


if annotation_template_path.is_file():
    annotation_template_df = pd.read_csv(
        annotation_template_path, dtype=str, keep_default_na=False
    )
    validate_columns(annotation_template_df, ANNOTATION_COLUMNS, "Annotation Template")
    template_action = "기존 파일 보존"
else:
    annotation_template_df = create_blank_query_slots()
    annotation_template_df.to_csv(
        annotation_template_path, index=False, encoding="utf-8-sig"
    )
    template_action = "신규 생성"

if annotations_long_path.is_file():
    annotations_long_df = pd.read_csv(
        annotations_long_path, dtype=str, keep_default_na=False
    )
    validate_columns(annotations_long_df, LONG_COLUMNS, "Long Annotation")
    long_action = "기존 파일 보존"
else:
    annotations_long_df = pd.DataFrame(columns=LONG_COLUMNS)
    annotations_long_df.to_csv(
        annotations_long_path, index=False, encoding="utf-8-sig"
    )
    long_action = "빈 구조 신규 생성"

if goldset_path.is_file():
    goldset_df = pd.read_csv(goldset_path, dtype=str, keep_default_na=False)
    validate_columns(goldset_df, GOLDSET_COLUMNS, "Goldset")
    goldset_action = "기존 파일 보존"
else:
    goldset_df = pd.DataFrame(columns=GOLDSET_COLUMNS)
    goldset_df.to_csv(goldset_path, index=False, encoding="utf-8-sig")
    goldset_action = "빈 구조 신규 생성"

print(f"Annotation Template: {template_action}")
print(f"Long Annotation: {long_action}")
print(f"Goldset: {goldset_action}")
display(annotation_template_df.groupby("query_type").size().to_frame("slot_count"))
display(annotation_template_df.head(12))


Annotation Template: 기존 파일 보존
Long Annotation: 기존 파일 보존
Goldset: 기존 파일 보존


,slot_count
query_type,
CONTEXT,18
DIRECT,24
MIXED,30
SCENE_ABSTRACT,24
SENSORY,24


,query_id,query_text,query_type,gold_accords,gold_notes,gold_seasons,gold_dayparts,gold_unresolved_expressions,resolution_status,annotator_confidence,annotator_comment
0,GQ0001,,DIRECT,,,,,,,,
1,GQ0002,,DIRECT,,,,,,,,
2,GQ0003,,DIRECT,,,,,,,,
3,GQ0004,,DIRECT,,,,,,,,
4,GQ0005,,DIRECT,,,,,,,,
5,GQ0006,,DIRECT,,,,,,,,
6,GQ0007,,DIRECT,,,,,,,,
7,GQ0008,,DIRECT,,,,,,,,
8,GQ0009,,DIRECT,,,,,,,,
9,GQ0010,,DIRECT,,,,,,,,


## 6. Annotation 기준

Gold label은 사용자가 **직접 말한 구조 Feature만** 기록한다. 의미를 과도하게 추론하지 않는다.

- “여름에 쓰기 좋은 citrus 향” → `accords=["citrus"]`, `seasons=["summer"]`, unresolved 없음
- “여름에 쓰기 좋은 상큼한 향” → `seasons=["summer"]`, unresolved=`["상큼한"]`
- “비 오는 숲 같은 향” → Feature 집합은 모두 `[]`, unresolved=`["비 오는 숲"]`
- “데이트할 때 좋은 향” → day/night로 바꾸지 않고 unresolved=`["데이트할 때"]`

“비 오는 숲 = woody + green”처럼 데이터 밖의 해석을 Gold에 넣지 않는다.

### Direct Mapping

- “우디한 향”은 직접적인 `woody` Accord로 label할 수 있다.
- “장미 향”처럼 Rose Note와 rose Accord가 모두 가능하면 하나를 강제하지 않는다. 허용 가능한 복수 label
  정책을 프로젝트에서 정하거나 `annotator_comment`에 모호성을 기록하고 Manual Review로 보낸다.

### Context

명시된 계절과 day/night만 label한다. 데이트, 출근, 파티 같은 상황을 시간대로 임의 변환하지 않는다.

### Unresolved

상큼한, 부드러운, 포근한, 차가운, 깨끗한, 무거운, 고급스러운, 섹시한, 도시적인, 자연스러운,
비 오는 숲, 바닷바람, 호텔 로비 등 현재 구조 Feature에 직접 대응하지 않는 표현을 그대로 기록한다.

### Annotator Confidence

Annotator Confidence
(정답을 붙인 사람이 자신의 판단을 얼마나 확신하는지 표시하는 값)는 `HIGH`, `MEDIUM`, `LOW` 중 하나다.
직접 표면형은 보통 HIGH, Accord/Note가 애매한 표현은 MEDIUM, 범위 자체가 불명확하면 LOW를 고려한다.


## 7. Annotation 파일 입력 형식

복수 값 column은 반드시 JSON array 문자열로 적는다.

```text
["citrus", "woody"]
[]
["비 오는 숲", "촉촉한"]
```

빈 문자열은 “아직 Annotation하지 않음”, `[]`는 “Annotation을 완료했고 해당 label이 없음”이다.

`resolution_status` 기준:

- `FULLY_RESOLVABLE`: unresolved가 없고 하나 이상의 구조 Feature가 있음
- `PARTIALLY_RESOLVABLE`: 구조 Feature와 unresolved 표현이 모두 있음
- `UNRESOLVABLE`: 구조 Feature가 없고 unresolved 표현이 하나 이상 있음

완전히 무의미하거나 평가가 불가능한 문장은 억지로 label하지 않고 검토 후 Goldset의 `review_status`를
`EXCLUDED`로 정한다.


In [3]:
guideline_text = "\n".join([
    "# Query Understanding Annotation Guideline",
    "",
    "## 목적",
    "",
    "이 데이터는 자연어 Query를 Accord, Note, Season, Daypart, unresolved expression으로 구조화하는 Stage 1만 평가합니다.",
    "향수 추천 결과나 Ranking의 정답을 만들지 않습니다.",
    "",
    "## Evaluation Leakage 방지",
    "",
    "Query 작성자는 11_rule_lexicon.csv, 11차 Parser 코드, Rule 목록, 11_parser_test_results.csv를 보지 않습니다.",
    "기존 Synthetic Query를 복사하거나 변형해 새 Query로 사용하지 않습니다.",
    "",
    "## 작업 순서",
    "",
    "1. 블라인드 Query 작성자가 12_query_annotation_template.csv의 query_text만 자유롭게 작성합니다.",
    "2. Annotator는 직접 언급된 Feature와 unresolved 표현을 JSON array 형식으로 기록합니다.",
    "3. 두 명 이상이 독립적으로 작업할 때는 12_query_annotations_long.csv에 실제 annotator_id와 label을 추가합니다.",
    "4. Agreement를 계산하되 자동 다수결로 Gold를 확정하지 않습니다.",
    "5. 사람이 의견 차이를 검토한 뒤 12_query_goldset.csv를 작성하고 READY / MANUAL_REVIEW / EXCLUDED를 표시합니다.",
    "",
    "## Query Type",
    "",
    "- DIRECT: Accord 또는 Note를 직접 언급",
    "- CONTEXT: Season 또는 Daypart를 명시",
    "- SENSORY: 감각적 형용사 중심",
    "- SCENE_ABSTRACT: 장면 또는 분위기 중심",
    "- MIXED: 직접 해석 가능한 표현과 어려운 표현이 함께 있음",
    "",
    "## Label 원칙",
    "",
    "- 사용자가 직접 말한 구조 Feature만 Gold로 기록합니다.",
    "- 계절과 day/night는 문장에 명시된 경우만 기록합니다.",
    "- 데이트, 출근, 파티를 day/night로 변환하지 않습니다.",
    "- 비 오는 숲을 woody, green 등으로 변환하지 않습니다.",
    "- Accord와 Note가 모두 가능한 표현은 annotator_comment에 모호성을 기록하고 Manual Review로 보냅니다.",
    "",
    "## JSON array 형식",
    "",
    "- 하나: [\"citrus\"]",
    "- 복수: [\"citrus\", \"woody\"]",
    "- 없음: []",
    "- 빈 문자열은 아직 Annotation하지 않았다는 뜻이므로 []와 다릅니다.",
    "",
    "## Resolution Status",
    "",
    "- FULLY_RESOLVABLE: 구조 Feature가 있고 unresolved 표현이 없음",
    "- PARTIALLY_RESOLVABLE: 구조 Feature와 unresolved 표현이 모두 있음",
    "- UNRESOLVABLE: 구조 Feature가 없고 unresolved 표현이 있음",
    "",
    "## Confidence",
    "",
    "- HIGH: 직접 대응이 명확함",
    "- MEDIUM: Accord/Note 선택 등 제한적인 모호성이 있음",
    "- LOW: 표현 범위 또는 label 판단이 매우 애매함",
    "",
    "## 예시",
    "",
    "- 여름에 쓰기 좋은 citrus 향 → accords=[\"citrus\"], seasons=[\"summer\"], unresolved=[]",
    "- 여름에 쓰기 좋은 상큼한 향 → seasons=[\"summer\"], unresolved=[\"상큼한\"]",
    "- 비 오는 숲 같은 향 → 모든 Feature=[], unresolved=[\"비 오는 숲\"]",
    "- 데이트할 때 좋은 향 → 모든 Feature=[], unresolved=[\"데이트할 때\"]",
    "",
    "## Gold 확정",
    "",
    "Agreement가 높아도 완전 자동 다수결로 확정하지 않습니다. Adjudicator가 원문과 모든 Annotation을 확인한 뒤",
    "최종 label을 기록합니다. 낮은 Agreement, 낮은 confidence, 애매한 Direct 표현은 반드시 Manual Review합니다.",
    "",
    "## Stage 2 금지",
    "",
    "이 Query Understanding Goldset에는 정답 향수 ID나 추천 적합도를 넣지 않습니다. 추천 Golden Set은 별도의",
    "독립적인 Human relevance 평가로 구축해야 합니다.",
    "",
    "## Circular Evaluation 방지",
    "",
    "현재 DB Feature를 이용해 추천 향수의 relevance 정답까지 자동 생성하지 않습니다.",
    "모델이 쓰는 기준으로 정답까지 만들면 같은 기준을 다시 맞히는 Circular Evaluation이 됩니다.",
    "",
])

if not guideline_path.is_file():
    guideline_path.write_text(guideline_text, encoding="utf-8")
    print(f"신규 생성: {guideline_path.name}")
else:
    print(f"기존 파일 보존: {guideline_path.name}")


기존 파일 보존: 12_annotation_guideline.md


## 8. 복수 Annotator 구조와 기본 검증

같은 Query를 두 명 이상이 독립적으로 label할 때 `12_query_annotations_long.csv`에 한 Annotator당 한 행을
추가한다. Notebook은 실제 `annotator_id`를 만들지 않는다.

아래 함수는 JSON array, 허용 값, Query ID, resolution 상태의 논리적 일관성을 검사한다. 오류를 자동 수정하지
않고 사람이 고칠 수 있도록 행별 문제를 반환한다.


In [4]:
LIST_FIELDS = [
    "gold_accords", "gold_notes", "gold_seasons", "gold_dayparts",
    "gold_unresolved_expressions",
]


def parse_json_list(value, allow_blank=False):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        if allow_blank:
            return None
        raise ValueError("빈 값입니다.")
    text = str(value).strip()
    if not text:
        if allow_blank:
            return None
        raise ValueError("빈 값입니다.")
    parsed = json.loads(text)
    if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):
        raise ValueError("문자열 JSON array가 아닙니다.")
    return parsed


def normalized_expression(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", text).strip()


def expected_resolution_status(row):
    feature_count = sum(len(parse_json_list(row[field])) for field in [
        "gold_accords", "gold_notes", "gold_seasons", "gold_dayparts"
    ])
    unresolved_count = len(parse_json_list(row["gold_unresolved_expressions"]))
    if feature_count > 0 and unresolved_count == 0:
        return "FULLY_RESOLVABLE"
    if feature_count > 0 and unresolved_count > 0:
        return "PARTIALLY_RESOLVABLE"
    if feature_count == 0 and unresolved_count > 0:
        return "UNRESOLVABLE"
    return None


def validate_annotation_rows(frame, require_annotator_id=False):
    issues = []
    known_query_ids = set(annotation_template_df["query_id"])
    for row_number, row in frame.reset_index(drop=True).iterrows():
        row_issues = []
        query_id = str(row.get("query_id", "")).strip()
        if query_id not in known_query_ids:
            row_issues.append("UNKNOWN_QUERY_ID")
        if require_annotator_id and not str(row.get("annotator_id", "")).strip():
            row_issues.append("MISSING_ANNOTATOR_ID")
        parsed_lists = {}
        for field in LIST_FIELDS:
            try:
                parsed_lists[field] = parse_json_list(row.get(field, ""))
            except Exception:
                row_issues.append(f"INVALID_JSON_ARRAY:{field}")
        status = str(row.get("resolution_status", "")).strip()
        if status not in ALLOWED_RESOLUTION_STATUS:
            row_issues.append("INVALID_RESOLUTION_STATUS")
        confidence = str(row.get("annotator_confidence", "")).strip()
        if confidence not in ALLOWED_CONFIDENCE:
            row_issues.append("INVALID_CONFIDENCE")
        if not any(issue.startswith("INVALID_JSON_ARRAY") for issue in row_issues):
            expected = expected_resolution_status(row)
            if expected is None:
                row_issues.append("NO_FEATURE_AND_NO_UNRESOLVED")
            elif status != expected:
                row_issues.append(f"STATUS_MISMATCH:expected={expected}")
        if row_issues:
            issues.append({
                "row_number": row_number,
                "query_id": query_id,
                "issues": " | ".join(row_issues),
            })
    if require_annotator_id and len(frame):
        duplicate_mask = frame.duplicated(
            ["query_id", "annotator_id"], keep=False
        )
        for row_number, row in frame[duplicate_mask].iterrows():
            issues.append({
                "row_number": row_number,
                "query_id": str(row.get("query_id", "")).strip(),
                "issues": "DUPLICATE_ANNOTATOR_FOR_QUERY",
            })
    return pd.DataFrame(issues, columns=["row_number", "query_id", "issues"])


long_annotation_issues_df = validate_annotation_rows(
    annotations_long_df, require_annotator_id=True
) if len(annotations_long_df) else pd.DataFrame(
    columns=["row_number", "query_id", "issues"]
)
display(long_annotation_issues_df)


,row_number,query_id,issues


## 9. Annotator Agreement 함수

Annotator Agreement
(여러 사람이 같은 Query를 비슷하게 해석했는지 보는 정도)는 두 Annotator의 모든 조합을 비교한다.

- Accord / Note: Jaccard Similarity
- Season / Daypart: Exact Match
- Resolution Status: Exact Match
- Unresolved expression: 정규화한 text 집합의 Jaccard Similarity

Jaccard Similarity
(두 집합의 합집합 중 교집합이 차지하는 비율)는 두 집합이 모두 비어 있으면 1로 정의한다.

전체 agreement는 여섯 component의 단순 평균이다. `ready_threshold=0.80`은 운영용 임시 검토 기준이며
품질 정답이 아니다. 낮은 점수는 `AMBIGUOUS`, 한 명뿐이면 `INSUFFICIENT_ANNOTATORS`로 표시한다.


In [5]:
def jaccard_similarity(left, right):
    left, right = set(left), set(right)
    union = left | right
    return len(left & right) / len(union) if union else 1.0


def row_label_sets(row):
    return {
        "accord": set(parse_json_list(row["gold_accords"])),
        "note": set(parse_json_list(row["gold_notes"])),
        "season": set(parse_json_list(row["gold_seasons"])),
        "daypart": set(parse_json_list(row["gold_dayparts"])),
        "unresolved": {
            normalized_expression(item)
            for item in parse_json_list(row["gold_unresolved_expressions"])
        },
        "resolution_status": str(row["resolution_status"]).strip(),
    }


def compute_annotator_agreement(long_frame, ready_threshold=0.80):
    if long_frame.empty:
        return pd.DataFrame(columns=[
            "query_id", "annotator_count", "annotator_pair_count",
            "accord_agreement", "note_agreement", "season_agreement",
            "daypart_agreement", "unresolved_agreement", "status_agreement",
            "agreement_score", "agreement_flag",
        ])

    issues = validate_annotation_rows(long_frame, require_annotator_id=True)
    if not issues.empty:
        raise ValueError("Annotation 오류를 먼저 수정하세요.\n" + issues.to_string(index=False))

    output_rows = []
    for query_id, group in long_frame.groupby("query_id", sort=True):
        group = group.drop_duplicates("annotator_id", keep=False)
        labels = [row_label_sets(row) for _, row in group.iterrows()]
        pairs = list(itertools.combinations(labels, 2))
        if not pairs:
            output_rows.append({
                "query_id": query_id,
                "annotator_count": len(group),
                "annotator_pair_count": 0,
                "accord_agreement": np.nan,
                "note_agreement": np.nan,
                "season_agreement": np.nan,
                "daypart_agreement": np.nan,
                "unresolved_agreement": np.nan,
                "status_agreement": np.nan,
                "agreement_score": np.nan,
                "agreement_flag": "INSUFFICIENT_ANNOTATORS",
            })
            continue

        component_scores = {
            "accord_agreement": [], "note_agreement": [],
            "season_agreement": [], "daypart_agreement": [],
            "unresolved_agreement": [], "status_agreement": [],
        }
        pair_overall_scores = []
        for left, right in pairs:
            scores = {
                "accord_agreement": jaccard_similarity(left["accord"], right["accord"]),
                "note_agreement": jaccard_similarity(left["note"], right["note"]),
                "season_agreement": float(left["season"] == right["season"]),
                "daypart_agreement": float(left["daypart"] == right["daypart"]),
                "unresolved_agreement": jaccard_similarity(
                    left["unresolved"], right["unresolved"]
                ),
                "status_agreement": float(
                    left["resolution_status"] == right["resolution_status"]
                ),
            }
            for key, value in scores.items():
                component_scores[key].append(value)
            pair_overall_scores.append(float(np.mean(list(scores.values()))))

        agreement_score = float(np.mean(pair_overall_scores))
        output_row = {
            "query_id": query_id,
            "annotator_count": len(group),
            "annotator_pair_count": len(pairs),
            **{
                key: float(np.mean(values))
                for key, values in component_scores.items()
            },
            "agreement_score": agreement_score,
            "agreement_flag": (
                "HIGH_AGREEMENT_CANDIDATE"
                if agreement_score >= ready_threshold
                else "AMBIGUOUS"
            ),
        }
        output_rows.append(output_row)
    return pd.DataFrame(output_rows)


agreement_df = compute_annotator_agreement(annotations_long_df)
display(agreement_df)

assert jaccard_similarity(set(), set()) == 1.0
assert jaccard_similarity({"a"}, {"a", "b"}) == 0.5


,query_id,annotator_count,annotator_pair_count,accord_agreement,note_agreement,season_agreement,daypart_agreement,unresolved_agreement,status_agreement,agreement_score,agreement_flag


## 10. Golden Label 확정과 Manual Review

Manual Review
(사람이 Annotator 의견 차이와 원문을 다시 보고 최종 정답을 결정하는 과정)는 모든 최종 Gold에 필요하다.
Agreement가 높아도 Notebook이 다수결로 label을 복사하지 않는다.

아래 함수는 검토 queue의 빈 Gold column과 agreement 점수만 준비한다. 모든 행은 처음에
`MANUAL_REVIEW`이며, Adjudicator가 원문과 Annotation을 확인한 뒤 label을 입력하고 `READY` 또는
`EXCLUDED`로 바꾼다.


In [6]:
def prepare_gold_review_queue(query_frame, agreement_frame):
    written_queries = query_frame[
        query_frame["query_text"].astype(str).str.strip().ne("")
    ][["query_id", "query_text", "query_type"]].copy()
    if written_queries.empty:
        return pd.DataFrame(columns=GOLDSET_COLUMNS)
    queue = written_queries.merge(
        agreement_frame[["query_id", "agreement_score"]]
        if not agreement_frame.empty
        else pd.DataFrame(columns=["query_id", "agreement_score"]),
        on="query_id",
        how="left",
    )
    for column in [
        "gold_accords", "gold_notes", "gold_seasons", "gold_dayparts",
        "gold_unresolved_expressions", "resolution_status",
    ]:
        queue[column] = ""
    queue["review_status"] = "MANUAL_REVIEW"
    return queue[GOLDSET_COLUMNS]


review_queue_preview_df = prepare_gold_review_queue(
    annotation_template_df, agreement_df
)
display(review_queue_preview_df)


,query_id,query_text,query_type,gold_accords,gold_notes,gold_seasons,gold_dayparts,gold_unresolved_expressions,resolution_status,agreement_score,review_status


## 11. Rule Parser 평가 Metric

Annotation이 완료된 `READY` Gold와 별도 Parser prediction이 있을 때만 평가한다.

- Exact Match Accuracy(모든 Feature 집합, unresolved 집합, resolution 상태를 전부 맞힌 Query 비율)
- Precision(모델이 추출한 항목 중 Gold에도 있는 비율)
- Recall(Gold 항목 중 모델이 실제로 추출한 비율)
- F1 Score(Precision과 Recall의 균형을 하나로 요약한 값)

Accord, Note, Season, Daypart, unresolved detection을 각각 계산한다. 분모가 0이면 억지로 0이나 1을
만들지 않고 `NaN`으로 둔다.

Prediction 파일의 예상 column:

`query_id`, `pred_accords`, `pred_notes`, `pred_seasons`, `pred_dayparts`,
`pred_unresolved_expressions`, `pred_resolution_status`


In [7]:
PREDICTION_COLUMNS = [
    "query_id", "pred_accords", "pred_notes", "pred_seasons",
    "pred_dayparts", "pred_unresolved_expressions", "pred_resolution_status",
]
FEATURE_SPECS = {
    "accord": ("gold_accords", "pred_accords", False),
    "note": ("gold_notes", "pred_notes", False),
    "season": ("gold_seasons", "pred_seasons", False),
    "daypart": ("gold_dayparts", "pred_dayparts", False),
    "unresolved": (
        "gold_unresolved_expressions", "pred_unresolved_expressions", True
    ),
}


def normalize_prediction_status(value):
    mapping = {
        "RESOLVED": "FULLY_RESOLVABLE",
        "PARTIALLY_RESOLVED": "PARTIALLY_RESOLVABLE",
        "UNRESOLVED": "UNRESOLVABLE",
    }
    text = str(value).strip()
    return mapping.get(text, text)


def value_set(value, normalize_items=False):
    items = parse_json_list(value)
    if normalize_items:
        return {normalized_expression(item) for item in items}
    return set(items)


def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    if np.isfinite(precision) and np.isfinite(recall) and precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    elif np.isfinite(precision) and np.isfinite(recall):
        f1 = 0.0
    else:
        f1 = np.nan
    return precision, recall, f1


def evaluate_subset(merged):
    if merged.empty:
        return {
            "query_count": 0, "exact_match_accuracy": np.nan,
            **{
                f"{feature}_{metric}": np.nan
                for feature in FEATURE_SPECS
                for metric in ("precision", "recall", "f1")
            },
        }

    exact_flags = []
    counts = {
        feature: {"tp": 0, "fp": 0, "fn": 0}
        for feature in FEATURE_SPECS
    }
    for row in merged.to_dict("records"):
        all_feature_exact = True
        for feature, (gold_column, pred_column, normalize_items) in FEATURE_SPECS.items():
            gold_values = value_set(row[gold_column], normalize_items)
            pred_values = value_set(row[pred_column], normalize_items)
            counts[feature]["tp"] += len(gold_values & pred_values)
            counts[feature]["fp"] += len(pred_values - gold_values)
            counts[feature]["fn"] += len(gold_values - pred_values)
            all_feature_exact &= gold_values == pred_values
        status_exact = (
            str(row["resolution_status"]).strip()
            == normalize_prediction_status(row["pred_resolution_status"])
        )
        exact_flags.append(all_feature_exact and status_exact)

    output = {
        "query_count": len(merged),
        "exact_match_accuracy": float(np.mean(exact_flags)),
    }
    for feature, values in counts.items():
        precision, recall, f1 = precision_recall_f1(
            values["tp"], values["fp"], values["fn"]
        )
        output.update({
            f"{feature}_precision": precision,
            f"{feature}_recall": recall,
            f"{feature}_f1": f1,
        })
    return output


def evaluate_rule_parser(gold_frame, prediction_frame):
    validate_columns(gold_frame, GOLDSET_COLUMNS, "Goldset")
    validate_columns(prediction_frame, PREDICTION_COLUMNS, "Prediction")
    ready_gold = gold_frame[gold_frame["review_status"].eq("READY")].copy()
    if ready_gold.empty:
        raise ValueError("READY Gold Query가 없습니다.")
    if ready_gold["query_id"].duplicated().any():
        raise ValueError("Goldset query_id가 중복되었습니다.")
    if prediction_frame["query_id"].duplicated().any():
        raise ValueError("Prediction query_id가 중복되었습니다.")
    merged = ready_gold.merge(
        prediction_frame, on="query_id", how="left", validate="one_to_one"
    )
    if merged[PREDICTION_COLUMNS[1:]].isna().any().any():
        missing = merged.loc[
            merged[PREDICTION_COLUMNS[1:]].isna().any(axis=1), "query_id"
        ].tolist()
        raise ValueError(f"Prediction이 없는 Query: {missing}")

    overall = pd.DataFrame([{
        "group_type": "OVERALL", "group_value": "ALL",
        **evaluate_subset(merged),
    }])
    by_query_type = pd.DataFrame([
        {
            "group_type": "QUERY_TYPE", "group_value": query_type,
            **evaluate_subset(group),
        }
        for query_type, group in merged.groupby("query_type", sort=True)
    ])
    by_resolution = pd.DataFrame([
        {
            "group_type": "RESOLUTION_STATUS", "group_value": status,
            **evaluate_subset(group),
        }
        for status, group in merged.groupby("resolution_status", sort=True)
    ])
    return pd.concat(
        [overall, by_query_type, by_resolution], ignore_index=True
    ), merged


assert precision_recall_f1(2, 0, 0) == (1.0, 1.0, 1.0)


## 12. Error Analysis 함수

Error Analysis
(모델이 틀린 사례를 직접 보고 원인을 분류하는 과정)는 한 Query에 여러 오류가 있으면 여러 행을 만든다.

- `MISSED_DIRECT_FEATURE`: Gold Accord/Note를 놓침
- `FALSE_FEATURE_EXTRACTION`: Gold에 없는 구조 Feature를 추출
- `MISSED_CONTEXT`: Gold Season/Daypart를 놓침
- `FAILED_UNRESOLVED_DETECTION`: Gold unresolved 표현을 찾지 못함
- `OVER_RESOLUTION`: 해석 근거가 없는 표현을 구조 Feature로 바꾸거나 완전 해결로 판단
- `UNDER_RESOLUTION`: 직접 이해 가능한 표현을 unresolved로 남기거나 미해결로 판단
- `AMBIGUOUS_QUERY`: Agreement가 낮아 Query 자체를 먼저 검토해야 함


In [8]:
def compact_label_json(row, prefix):
    return json.dumps({
        "accords": row[f"{prefix}_accords"],
        "notes": row[f"{prefix}_notes"],
        "seasons": row[f"{prefix}_seasons"],
        "dayparts": row[f"{prefix}_dayparts"],
        "unresolved": row[f"{prefix}_unresolved_expressions"],
        "resolution_status": (
            row["resolution_status"]
            if prefix == "gold"
            else row["pred_resolution_status"]
        ),
    }, ensure_ascii=False)


def build_error_analysis(merged, agreement_frame=None):
    agreement_lookup = {}
    if agreement_frame is not None and not agreement_frame.empty:
        agreement_lookup = agreement_frame.set_index("query_id")[
            "agreement_flag"
        ].to_dict()
    errors = []
    for row in merged.to_dict("records"):
        gold_sets = {
            feature: value_set(row[gold_column], normalize_items)
            for feature, (gold_column, _, normalize_items) in FEATURE_SPECS.items()
        }
        pred_sets = {
            feature: value_set(row[pred_column], normalize_items)
            for feature, (_, pred_column, normalize_items) in FEATURE_SPECS.items()
        }
        error_items = []
        missed_direct = (
            (gold_sets["accord"] - pred_sets["accord"])
            | (gold_sets["note"] - pred_sets["note"])
        )
        if missed_direct:
            error_items.append((
                "MISSED_DIRECT_FEATURE",
                f"놓친 Accord/Note: {sorted(missed_direct)}",
            ))
        false_features = set().union(*[
            pred_sets[name] - gold_sets[name]
            for name in ("accord", "note", "season", "daypart")
        ])
        if false_features:
            error_items.append((
                "FALSE_FEATURE_EXTRACTION",
                f"Gold에 없는 구조 Feature: {sorted(false_features)}",
            ))
        missed_context = (
            (gold_sets["season"] - pred_sets["season"])
            | (gold_sets["daypart"] - pred_sets["daypart"])
        )
        if missed_context:
            error_items.append((
                "MISSED_CONTEXT",
                f"놓친 Season/Daypart: {sorted(missed_context)}",
            ))
        missed_unresolved = gold_sets["unresolved"] - pred_sets["unresolved"]
        if missed_unresolved:
            error_items.append((
                "FAILED_UNRESOLVED_DETECTION",
                f"놓친 unresolved 표현: {sorted(missed_unresolved)}",
            ))
        gold_status = str(row["resolution_status"]).strip()
        pred_status = normalize_prediction_status(row["pred_resolution_status"])
        if gold_status in ("PARTIALLY_RESOLVABLE", "UNRESOLVABLE") and (
            pred_status == "FULLY_RESOLVABLE" or false_features
        ):
            error_items.append((
                "OVER_RESOLUTION",
                "Gold는 unresolved 의도가 있으나 Parser가 완전 해결했거나 추가 Feature를 만들었습니다.",
            ))
        if gold_status == "FULLY_RESOLVABLE" and (
            pred_status != "FULLY_RESOLVABLE" or pred_sets["unresolved"]
        ):
            error_items.append((
                "UNDER_RESOLUTION",
                "Gold는 완전 해결 가능하지만 Parser가 미해결 상태 또는 unresolved 표현을 만들었습니다.",
            ))
        if agreement_lookup.get(row["query_id"]) == "AMBIGUOUS":
            error_items.append((
                "AMBIGUOUS_QUERY",
                "Annotator Agreement가 낮아 모델 오류보다 Gold 해석을 먼저 검토해야 합니다.",
            ))

        gold_json = compact_label_json(row, "gold")
        prediction_json = compact_label_json(row, "pred")
        for error_type, description in error_items:
            errors.append({
                "query": row["query_text"],
                "query_type": row["query_type"],
                "gold": gold_json,
                "prediction": prediction_json,
                "error_type": error_type,
                "error_description": description,
            })
    return pd.DataFrame(errors, columns=[
        "query", "query_type", "gold", "prediction",
        "error_type", "error_description",
    ])


## 13. Query Type별·Resolution별 평가

`evaluate_rule_parser`는 전체 평균뿐 아니라 실제 Gold에 존재하는 다음 그룹을 자동으로 출력한다.

- Query Type: `DIRECT`, `CONTEXT`, `SENSORY`, `SCENE_ABSTRACT`, `MIXED`
- Gold status: `FULLY_RESOLVABLE`, `PARTIALLY_RESOLVABLE`, `UNRESOLVABLE`

Direct/Context가 잘하고 Sensory/Scene이 못할 것이라고 미리 결론 내리지 않는다. 특히 “모르는 것을 모른다고
말하는 능력”은 unresolved Precision/Recall/F1과 status 포함 Exact Match로 실제 label에서 확인한다.


## 14. 현재 실행 상태와 조건부 평가

실제 사람이 채운 `READY` Gold와 `12_rule_parser_predictions.csv`가 모두 있을 때만 평가 CSV를 만든다.
그렇지 않으면 가짜 점수를 만들지 않고 `WAITING_FOR_HUMAN_ANNOTATION`으로 종료한다.


In [9]:
filled_query_mask = annotation_template_df["query_text"].astype(str).str.strip().ne("")
filled_query_count = int(filled_query_mask.sum())
annotator_count = int(
    annotations_long_df["annotator_id"].astype(str).str.strip().replace("", np.nan).nunique()
) if len(annotations_long_df) else 0
ready_gold_count = int(goldset_df["review_status"].eq("READY").sum()) if len(goldset_df) else 0
manual_review_count = int(
    goldset_df["review_status"].eq("MANUAL_REVIEW").sum()
) if len(goldset_df) else 0
mean_agreement = float(agreement_df["agreement_score"].mean()) if (
    len(agreement_df) and agreement_df["agreement_score"].notna().any()
) else np.nan

if ready_gold_count > 0 and prediction_path.is_file():
    predictions_df = pd.read_csv(prediction_path, dtype=str, keep_default_na=False)
    evaluation_df, evaluation_merged_df = evaluate_rule_parser(
        goldset_df, predictions_df
    )
    errors_df = build_error_analysis(evaluation_merged_df, agreement_df)
    evaluation_df.to_csv(evaluation_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")
    notebook_status = "EVALUATION_COMPLETE"
    display(evaluation_df)
    display(errors_df)
else:
    evaluation_df = pd.DataFrame()
    errors_df = pd.DataFrame()
    notebook_status = "WAITING_FOR_HUMAN_ANNOTATION"
    display(Markdown(
        "# STATUS: WAITING_FOR_HUMAN_ANNOTATION\n\n"
        "실제 사람이 Query와 Gold label을 작성하지 않았으므로 Rule Parser 평가 점수를 생성하지 않았습니다."
    ))

current_status_df = pd.Series({
    "template_slots": len(annotation_template_df),
    "written_queries": filled_query_count,
    "annotators": annotator_count,
    "agreement_mean": mean_agreement,
    "ready_gold_queries": ready_gold_count,
    "manual_review_queries": manual_review_count,
    "status": notebook_status,
}, name="value")
display(current_status_df.to_frame())


# STATUS: WAITING_FOR_HUMAN_ANNOTATION

실제 사람이 Query와 Gold label을 작성하지 않았으므로 Rule Parser 평가 점수를 생성하지 않았습니다.

,value
template_slots,120
written_queries,0
annotators,0
agreement_mean,NaN
ready_gold_queries,0
manual_review_queries,0
status,WAITING_FOR_HUMAN_ANNOTATION


## 15. Circular Evaluation 방지

이번 Gold는 “여름 citrus 향 → citrus Accord + summer”까지 평가한다. 어떤 향수 ID가 정답인지는 정하지
않는다.

Circular Evaluation
(모델이 사용하는 Feature로 추천 정답까지 만든 뒤 같은 모델이 그 정답을 잘 맞힌다고 평가하는 문제)을
피하려면 Stage 2의 향수 relevance는 현재 DB Feature와 독립적인 사람의 판단에서 와야 한다.


## 16. Recommendation Golden Set 설계 초안

Stage 2에서는 실제 사람이 Query와 후보 향수의 적합도를 평가한다.

Relevance Grade
(Query와 추천 향수가 얼마나 잘 맞는지 사람이 단계별 점수로 평가한 값)의 예시는 다음과 같다.

- 0: 전혀 적합하지 않음
- 1: 조금 적합
- 2: 적합
- 3: 매우 적합

현재 DB의 Accord/Note/Season 값만으로 Grade를 자동 생성하지 않는다. 가능하면 실제 시향, 충분한 제품 설명,
독립적인 다수 평가자의 판단과 confidence를 함께 수집한다.

### Candidate Pool

Candidate Pool
(사람이 실제 평가할 수 있도록 여러 검색 방식에서 모은 제한된 향수 후보 집합)은 한 모델의 TOP 결과로만
만들지 않는다.

Query마다 중복을 제거해 다음 출처를 섞는다.

- Rule Baseline TOP N
- 구조 기반 Retrieval TOP N
- Community Reranking 후보
- Popularity 후보
- 일부 Random 후보

각 후보에 `source_systems`를 기록하고 평가자에게 출처와 rank를 숨긴다. 한 시스템의 후보만 쓰면 그 시스템이
찾을 수 없는 향수가 Gold 후보에 들어오지 않아 해당 모델에 유리한 평가가 된다.


## 17. Golden Set Split 설계

충분한 Query와 유형별 label이 확보된 뒤 Query 단위로 다음처럼 나눈다.

- Development(모델을 수정하면서 반복적으로 확인해도 되는 데이터): 약 80%
- Final Holdout(모델 개발에는 사용하지 않고 마지막 평가에만 한 번 사용하는 데이터): 약 20%

같은 Query의 paraphrase나 매우 유사한 문장은 반드시 같은 split에 둬야 한다. Query 수가 너무 적거나 유형별
분포가 불안정하면 80/20을 억지로 나누지 말고 먼저 더 수집한다. Final Holdout은 Parser 규칙, LLM prompt,
threshold 결정에 사용하지 않는다.


## 18. 생성 파일 검증

필수 네 파일의 구조와 현재 비어 있어야 하는 값을 확인한다. Annotation 완료 전에는 평가 결과 파일을 요구하지
않는다.


In [10]:
output_audit_rows = []
for path, expected_columns in [
    (annotation_template_path, ANNOTATION_COLUMNS),
    (annotations_long_path, LONG_COLUMNS),
    (goldset_path, GOLDSET_COLUMNS),
]:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    validate_columns(frame, expected_columns, path.name)
    output_audit_rows.append({
        "file": path.name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "size_kb": path.stat().st_size / 1024,
    })
if not guideline_path.is_file():
    raise FileNotFoundError(guideline_path)
output_audit_rows.append({
    "file": guideline_path.name,
    "rows": len(guideline_path.read_text(encoding="utf-8").splitlines()),
    "columns": np.nan,
    "size_kb": guideline_path.stat().st_size / 1024,
})
output_audit_df = pd.DataFrame(output_audit_rows)
display(output_audit_df)

if notebook_status == "WAITING_FOR_HUMAN_ANNOTATION":
    assert evaluation_df.empty and errors_df.empty


,file,rows,columns,size_kb
0,12_query_annotation_template.csv,120,11.000000,3.178711
1,12_query_annotations_long.csv,0,10.000000,0.158203
2,12_query_goldset.csv,0,11.000000,0.158203
3,12_annotation_guideline.md,76,NaN,3.500977


# 12차 분석 결론

## 1. 이번 Golden Set의 목적

실제 사람이 자유롭게 작성한 자연어의 **Query Understanding** 품질을 평가하기 위한 독립적인 Stage 1
Golden Set을 설계했다.

## 2. 평가하는 것

- Accord
- Note
- Season
- Daypart
- Unresolved Expression
- Partial Resolution과 resolution status

## 3. 평가하지 않는 것

최종 향수 추천 품질, TOP 10 Ranking, 사용자 만족도는 이번 Gold의 평가 대상이 아니다.

## 4. 기존 Synthetic Query와의 차이

기존 126개 Query는 Rule Lexicon과 Template으로 만든 개발용 동작 확인 데이터다. 새 Query는 작성자가
Rule Lexicon과 Parser를 보지 않은 상태에서 자유롭게 쓰며, 기존 문장을 포함하거나 변형하지 않는다.

## 5. Annotation 현황

- Query 수: **0개 작성 / 120개 빈 슬롯**
- Annotator 수: **0명**
- Agreement: **N/A**
- Manual Review 수: **0개**

## 6. Rule Parser 실제 평가

**WAITING_FOR_HUMAN_ANNOTATION**

실제 사람이 작성한 Query와 확정 Gold가 없으므로 Exact Match, Accord/Note/Season/Daypart F1,
Unresolved F1을 계산하지 않았다. 가짜 점수나 기존 Synthetic 결과를 대신 넣지 않았다.

## 7. 가장 많이 발생한 오류

Annotation과 Parser prediction이 없으므로 아직 집계할 수 없다. 완료 후 Error Analysis 함수가 오류 유형별
사례 표를 만든다.

## 8. LLM이 필요한 영역

아직 결론 내릴 수 없다. 실제 Gold에서 Query Type별·resolution status별 Rule Parser 결과와 오류를 확인한
뒤, 어떤 표현에서 외부 의미 해석이 필요한지 판단한다.

## 9. Recommendation 평가의 남은 문제

실제 향수 적합성에 대한 독립적인 Human Label이 필요하다. 현재 DB Feature로 추천 Grade를 자동 생성하면
Circular Evaluation이 되므로 별도 Candidate Pool과 blinded relevance 평가가 필요하다.

## 10. 다음 단계

Golden Set Annotation 완료

↓

Rule Parser 성능 확정

↓

LLM Query Parser 구현

↓

동일 Golden Set에서 Rule vs LLM 비교
